# Fine Tuning XLRS to Japanese and US English


**Author:**

Salvador Wahnon Palma s2665070@u.tsukuba.ac.jp

University of Tsukuba / Interaction Lab


<br>

**Objective:**
 
Generates L1-Informed vs Articulatory corrective feedback for English speakers learning Japanese pronunciation.

---

# 1. Setup Drive

In [6]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [7]:
import pandas as pd

us_train = pd.read_parquet('drive/MyDrive/Tsukuba/Datasets and Models/US_train.parquet')
us_validation = pd.read_parquet('drive/MyDrive/Tsukuba/Datasets and Models/US_validation.parquet')
us_test = pd.read_parquet('drive/MyDrive/Tsukuba/Datasets and Models/US_test.parquet')

jp_train = pd.read_parquet('drive/MyDrive/Tsukuba/Datasets and Models/JP_train.parquet')
jp_validation = pd.read_parquet('drive/MyDrive/Tsukuba/Datasets and Models/JP_validation.parquet')
jp_test = pd.read_parquet('drive/MyDrive/Tsukuba/Datasets and Models/JP_test.parquet')

In [9]:
print(f"JP Train shape: {jp_train.shape}, JP Validation shape: {jp_validation.shape}")
print(f"US Train shape: {us_train.shape}, US Validation shape: {us_validation.shape}")

print(us_train.columns)
print(type(us_train.iloc[0]["audio"]))
print(us_train.iloc[0]["audio"])
print(type(us_train.iloc[0]["IPA"]))
print(us_train.iloc[0]["IPA"])

print(jp_train.columns)
print(type(jp_train.iloc[0]["audio"]))
print(jp_train.iloc[0]["audio"])
print(type(jp_train.iloc[0]["IPA"]))
print(jp_train.iloc[23]["IPA"])

JP Train shape: (7840, 2), JP Validation shape: (882, 2)
US Train shape: (3629, 2), US Validation shape: (670, 2)
Index(['audio', 'IPA'], dtype='object')
<class 'dict'>
{'bytes': b'RIFF$8\x01\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00data\x008\x01\x00\xf9\xff\x02\x00\x01\x00\x01\x00\xfd\xff\x00\x00\xff\xff\x01\x00\x02\x00\x01\x00\xff\xff\x03\x00\x02\x00\x00\x00\x03\x00\x02\x00\x02\x00\x02\x00\x02\x00\x00\x00\x00\x00\x02\x00\xff\xff\x01\x00\x01\x00\x01\x00\xff\xff\x01\x00\x00\x00\x00\x00\x01\x00\x01\x00\xff\xff\x00\x00\x01\x00\xff\xff\x02\x00\x01\x00\xfe\xff\xfe\xff\x00\x00\x02\x00\xff\xff\x02\x00\xff\xff\xfe\xff\xfe\xff\x01\x00\xff\xff\x01\x00\xfe\xff\xff\xff\xfe\xff\x00\x00\xff\xff\xff\xff\x01\x00\x02\x00\x00\x00\x01\x00\x02\x00\x00\x00\x00\x00\x01\x00\x00\x00\x02\x00\x01\x00\x00\x00\x02\x00\x00\x00\xff\xff\x01\x00\x03\x00\x01\x00\xfd\xff\xff\xff\x01\x00\x05\x00\x08\x00\xfc\xff\xfe\xff\xfd\xff\x01\x00\x00\x00\xfb\xff\x08\x00\x06\x00\x02\x00\x

In [10]:


# for i in range(len(jp_test)):
#     phonemes = jp_test.iloc[i]["IPA"].split()
#     result = []
#     for phoneme in phonemes:
#         if phoneme == "pyː":
#             result.append("pː")
#             result.append("j")
#         elif phoneme == "kyː":
#             result.append("kː")
#             result.append("j")
#         else:
#             result.append(phoneme)
#     jp_test.iloc[i]["IPA"] = " ".join(result)
       
# jp_test.to_parquet("drive/MyDrive/Tsukuba/Datasets and Models/JP_test.parquet")     

us_phonemes = {symbol for seq in us_train["IPA"].dropna() for symbol in seq.split()}
jp_phonemes = {symbol for seq in jp_train["IPA"].dropna() for symbol in seq.split()}

print(us_phonemes)
print(jp_phonemes)
    

{'r', 'θ', 'ɾ', 'ɚ', 'ɪ', 'ʒ', 'p', 'tʃ', 'j', 'z', 'ɔ', 'h', 'aɪ', 'm', 'ð', 'g', 'eɪ', 'ʌ', 'oʊ', 'ʔ', 'u', 'b', 'l', 'ɛ˞', 'ŋ', 'ɑ', 'ə', 'ʊ', 'ɛ', 's', 'f', 'æ', 'v', 't', 'ʃ', 'w', 'i', 'aʊ', 'dʒ', 'd', 'k', 'ɔɪ', 'n'}
{'ts', 'tɕ', 'r', 'ɾ', 'ɯ', 'tː', 'ɡː', 'ɕː', 'a', 'tsː', 'ç', 'kʲ', 'p', 'j', 'mː', 'z', 'o', 'h', 'pː', 'm', 'dʑː', 'ɕ', 'g', 'kː', 'ɸ', 'sː', 'dː', 'b', 'i̥', 'ŋ', 'ɡ', 'ɯ̥', 'tɕː', 's', 'hː', 'e', 'dʑ', 'v', 't', 'w', 'i', 'd', 'k', 'n', 'nː'}


# 2. Fine-Tune XLRS

In [11]:
# The Colab image has a BROKEN datasets/pyarrow pair: `import pyarrow.json` fails with
# "cannot import name 'open_json'". We don't use `datasets`, BUT transformers.Trainer does
# `if is_datasets_available(): import datasets` internally, so the broken package crashes
# `from transformers import Trainer`. Fix at the package level: remove datasets so
# is_datasets_available() is False and Trainer skips that import; reinstall a matching
# pyarrow so pandas parquet reads stay healthy.
#
# IMPORTANT: after this cell runs, do Runtime > Restart session, then run from cell 2.0
# (skip this install cell on the second pass). One restart is required for the uninstall
# to take effect in the live kernel.
!pip uninstall -y -q datasets 2>/dev/null
!pip install -q -U "transformers>=4.44" "accelerate>=0.33" jiwer soundfile "pyarrow>=17,<19"
print("Done. Now: Runtime > Restart session, then run from cell 2.0 (Setup Drive).")


Done. Now: Runtime > Restart session, then run from cell 2.0 (Setup Drive).


In [12]:
# --- 2.0 Imports (no `datasets`) ---
# transformers.Trainer does `if is_datasets_available(): import datasets` internally, and
# on this image that import crashes (broken pyarrow). is_datasets_available() uses
# importlib.util.find_spec, so the ONLY reliable fix is that `datasets` is actually
# uninstalled (done in the install cell) + a runtime restart. This probe confirms it:
import importlib.util
if importlib.util.find_spec("datasets") is not None:
    raise RuntimeError(
        "`datasets` is still installed -> transformers.Trainer will crash on the broken "
        "pyarrow. Run the install cell above (it uninstalls datasets), then "
        "Runtime > Restart session, then run from here."
    )

import io
import numpy as np
import soundfile as sf
import torch
from transformers import Trainer as _TrainerProbe  # must succeed now  # noqa: F401

print("datasets absent -> Trainer import OK")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

XLSR_MODEL_ID = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"
OUTPUT_DIR = "/content/drive/MyDrive/Tsukuba/Datasets and Models/xlsr-jp-us-ipa"


datasets absent -> Trainer import OK
CUDA available: True
GPU: Tesla T4


## 2.1 Combine US + JP into one DataFrame per split

The parquet files are pandas DataFrames with `audio` (dict: `bytes` = WAV, `path`) and a
space-separated `IPA` string. We just tag each with `lang` and concatenate US + JP — no
`datasets` library. Decoding happens later, per-sample, inside the PyTorch `Dataset`.

In [13]:
def CombineSplits(us_df, jp_df):
    us = us_df[["audio", "IPA"]].copy(); us["lang"] = "us"
    jp = jp_df[["audio", "IPA"]].copy(); jp["lang"] = "jp"
    return pd.concat([us, jp], ignore_index=True)

train_df = CombineSplits(us_train, jp_train).sample(frac=1, random_state=42).reset_index(drop=True)
eval_df = CombineSplits(us_validation, jp_validation).reset_index(drop=True)

print("Train:", len(train_df), "| Eval:", len(eval_df))

# Peek: decode one clip's WAV bytes so we can confirm sample rate + shape.
_row = train_df.iloc[0]
_arr, _sr = sf.read(io.BytesIO(_row["audio"]["bytes"]), dtype="float32")
print("lang:", _row["lang"])
print("IPA :", _row["IPA"])
print("audio sr:", _sr, "| samples:", _arr.shape)


Train: 11469 | Eval: 1552
lang: us
IPA : p ɛ ŋ w ɪ n z l ɪ v n ɪ ɛ˞ ð ʌ ʔ aɪ s i æ n ɑ r ɾ ɪ k
audio sr: 16000 | samples: (43725,)


## 2.2 Build the CTC vocabulary + processor

The IPA labels are **multi-character** tokens (`tʃ`, `dʑː`, `ɯ̥`, `eɪ`, …) separated by
spaces. `Wav2Vec2CTCTokenizer` matches greedily by character, so a raw space-separated
string glues adjacent symbols together. We instead build the vocab from the union of the
US + JP phoneme inventories, use `|` as an explicit `word_delimiter_token` between **every**
phoneme, and decode ids → phonemes directly (`IdsToIpa`). A round-trip assert proves the
encoding is lossless, including repeated long-vowel phonemes.

We **discard the pretrained head/tokenizer** (its eSpeak vocab doesn't match our custom JP
symbols) and keep only the pretrained `Wav2Vec2` encoder — a new CTC head is trained on top.

In [14]:
import json
import os
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# Union of phoneme inventories across BOTH train + eval splits (pandas columns now).
all_phonemes = set()
for seq in pd.concat([train_df["IPA"], eval_df["IPA"]]).dropna():
    all_phonemes.update(seq.split())

# The IPA symbols are MULTI-CHARACTER (tʃ, dʑː, ɯ̥, eɪ, ...). Wav2Vec2CTCTokenizer
# does greedy character matching, so if we let it parse a raw space-separated string
# it glues adjacent multi-char symbols together (e.g. "m i kː ɯ" -> "m ikːɯ").
# To keep every phoneme a distinct CTC unit we use "|" as the word-delimiter token and
# join phonemes with "|" before tokenizing, so the tokenizer sees an explicit boundary
# between EVERY phoneme. "|" never occurs inside an IPA symbol, so it is unambiguous.
DELIM = "|"
assert DELIM not in all_phonemes

vocab_list = sorted(all_phonemes)
vocab_dict = {p: i for i, p in enumerate(vocab_list)}
vocab_dict[DELIM] = len(vocab_dict)          # word-delimiter token
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)        # PAD doubles as the CTC blank token

print(f"Vocab size: {len(vocab_dict)} (= {len(vocab_list)} phonemes + delim + UNK + PAD)")

os.makedirs(OUTPUT_DIR, exist_ok=True)
VOCAB_PATH = os.path.join(OUTPUT_DIR, "vocab.json")
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token=DELIM,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(OUTPUT_DIR)


def IpaToDelimited(ipa: str) -> str:
    """'m i kː ɯ' -> 'm|i|kː|ɯ' so each phoneme is a separate CTC unit."""
    return DELIM.join(ipa.split())


# NOTE: tokenizer.decode() strips the word-delimiter token and space-joins the rest,
# which glues multi-char phonemes back together. So we DON'T decode to a string and
# re-parse; we map ids -> tokens directly and keep the delimiter as the separator.
_SPECIAL = {tokenizer.pad_token, tokenizer.unk_token,
            getattr(tokenizer, "bos_token", None), getattr(tokenizer, "eos_token", None)}

def IdsToIpa(ids, group_tokens: bool = True) -> str:
    """CTC ids -> 'm i kː ɯ'.

    group_tokens collapses consecutive duplicate ids (standard CTC decoding of model
    predictions). Genuine repeated phonemes (e.g. the 'a a' long vowels in JVS) survive
    because IpaToDelimited puts a DELIM id between them, so they are never *consecutive*
    identical ids. Blank/pad, UNK, and the delimiter are dropped from the output.
    """
    ids = [int(i) for i in ids]
    if group_tokens:
        ids = [i for j, i in enumerate(ids) if j == 0 or i != ids[j - 1]]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    phonemes = [t for t in tokens if t not in _SPECIAL and t != DELIM]
    return " ".join(phonemes)


# Sanity check: encode -> id-decode round-trip must reproduce the sequence EXACTLY,
# including repeated phonemes like the 'a a' / 'i i' long vowels in JVS.
_ex_ipa = train_df.loc[train_df["lang"] == "jp", "IPA"].iloc[0]
_ids = tokenizer(IpaToDelimited(_ex_ipa)).input_ids
_ungrouped = IdsToIpa(_ids, group_tokens=False)
_grouped = IdsToIpa(_ids, group_tokens=True)
print("IPA      :", _ex_ipa)
print("decoded  :", _grouped)
assert _ungrouped == _ex_ipa, f"Ungrouped round-trip lossy: {_ungrouped}"
assert _grouped == _ex_ipa, f"Grouped decode dropped repeats: {_grouped}"
print("Round-trip OK (grouped and ungrouped both exact)")


Vocab size: 73 (= 70 phonemes + delim + UNK + PAD)
IPA      : k e e m j o o ɕ a d a ts ɯ n a n a ɾ e e ɕ o ŋ k a ɾ a dʑ o o tɕ o k a n a ɸ ɯ ɾ e ɾ ɯ k a t a ɾ i m a d e h a b a h i ɾ o i ç j o o ɡ e n r j o k ɯ o m o ts ɯ̥
decoded  : k e e m j o o ɕ a d a ts ɯ n a n a ɾ e e ɕ o ŋ k a ɾ a dʑ o o tɕ o k a n a ɸ ɯ ɾ e ɾ ɯ k a t a ɾ i m a d e h a b a h i ɾ o i ç j o o ɡ e n r j o k ɯ o m o ts ɯ̥
Round-trip OK (grouped and ungrouped both exact)


## 2.3 PyTorch Dataset + CTC collator

A `torch.utils.data.Dataset` is all the model needs. `__getitem__` returns one sample as
`{input_values, labels}`:

- **audio** → decode WAV bytes (`soundfile`) → feature-extractor normalize → `input_values`
- **IPA** → `IpaToDelimited` → tokenize → `labels`

The **collator** pads a list of these into batched tensors (audio padded with 0, labels
with -100 so CTC ignores them). `Trainer` builds the `DataLoader`, calls the collator, and
moves each batch to the GPU — we write no device code. This is the standard HF wav2vec2
fine-tuning recipe, with zero dependency on the (broken) `datasets` library.

In [15]:
from dataclasses import dataclass
from typing import Dict, List, Union
from torch.utils.data import Dataset as TorchDataset

# Cap clip length to bound peak GPU memory. wav2vec2 self-attention memory scales with
# sequence_length**2, so one very long clip in a batch blows up VRAM. 12 s @ 16 kHz is a
# safe ceiling for a 15 GB T4 and drops only a tiny tail of clips.
MAX_AUDIO_SECONDS = 12
MAX_SAMPLES = MAX_AUDIO_SECONDS * 16000

def AudioNumSamples(cell):
    # Read only the WAV header (soundfile.info) — no full decode — to get length cheaply.
    return sf.info(io.BytesIO(cell["bytes"])).frames

def FilterLong(df, name):
    n = df["audio"].map(AudioNumSamples)
    keep = n <= MAX_SAMPLES
    print(f"{name}: keeping {keep.sum()}/{len(df)} clips (<= {MAX_AUDIO_SECONDS}s); "
          f"dropped {(~keep).sum()}")
    return df[keep].reset_index(drop=True)

train_df = FilterLong(train_df, "train")
eval_df = FilterLong(eval_df, "eval")


class SpeechDataset(TorchDataset):
    """One row -> {input_values: float32 np.array, labels: list[int]}.

    Decodes audio and tokenizes IPA lazily per sample (cheap; happens inside DataLoader
    workers). No global state beyond the shared, read-only `processor`.
    """
    def __init__(self, df):
        self.audio = list(df["audio"])   # list of {"bytes":..., "path":...}
        self.ipa = list(df["IPA"])

    def __len__(self):
        return len(self.ipa)

    def __getitem__(self, i):
        array, sr = sf.read(io.BytesIO(self.audio[i]["bytes"]), dtype="float32")
        if array.ndim > 1:               # safety: downmix if ever stereo
            array = array.mean(axis=1)
        input_values = processor(array, sampling_rate=sr).input_values[0]
        labels = processor(text=IpaToDelimited(self.ipa[i])).input_ids
        return {"input_values": input_values, "labels": labels}


train_dataset = SpeechDataset(train_df)
eval_dataset = SpeechDataset(eval_df)

# Smoke-test one sample so any decode/tokenize bug surfaces HERE, not mid-training.
_s = train_dataset[0]
print("input_values shape:", np.asarray(_s["input_values"]).shape)
print("labels[:12]       :", _s["labels"][:12])
print("train:", len(train_dataset), "| eval:", len(eval_dataset))


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        # Replace padding with -100 so CTC loss ignores it.
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)


train: keeping 11063/11469 clips (<= 12s); dropped 406
eval: keeping 1492/1552 clips (<= 12s); dropped 60
input_values shape: (43725,)
labels[:12]       : [28, 70, 55, 70, 47, 70, 42, 70, 59, 70, 24, 70]
train: 11063 | eval: 1492


## 2.4 Metric (PER), model, and Trainer

**PER** (Phoneme Error Rate) = word-error-rate computed over space-separated phonemes. Since our decoded strings are already space-separated phonemes, `jiwer.wer` on them is exactly PER.

The encoder is loaded from `facebook/wav2vec2-xlsr-53-espeak-cv-ft` with a fresh CTC head sized to our vocab. We freeze the CNN feature extractor (standard for wav2vec2 fine-tuning).

In [16]:
import jiwer
from transformers import Wav2Vec2ForCTC

def ComputeMetrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Map ids -> phonemes directly (see IdsToIpa in 2.2). Predictions are CTC output,
    # so group_tokens=True (collapse blanks/repeats); labels keep every token.
    pred_str = [IdsToIpa(row, group_tokens=True) for row in pred_ids]
    label_str = [IdsToIpa(row, group_tokens=False) for row in label_ids]

    # Guard against empty refs (jiwer errors on empty strings).
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"per": 1.0}
    preds, refs = zip(*pairs)
    per = jiwer.wer(list(refs), list(preds))
    return {"per": per}

model = Wav2Vec2ForCTC.from_pretrained(
    XLSR_MODEL_ID,
    attention_dropout=0.05,
    hidden_dropout=0.05,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.05,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,   # new CTC head shape differs from pretrained
)
model.freeze_feature_encoder()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xlsr-53-espeak-cv-ft
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([392, 1024]) vs model:torch.Size([75, 1024])
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([392]) vs model:torch.Size([75])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
import inspect
from transformers import TrainingArguments, Trainer

use_cuda = torch.cuda.is_available()

# Checkpoints go to Colab's LOCAL disk (/content, ~100 GB, ephemeral), NOT Drive. Each
# checkpoint is ~1.2 GB (with optimizer state) and the Trainer keeps best+latest, which
# previously filled the small Drive quota mid-run and silently dropped later saves. Local
# disk has ample room; we copy only the final best model (~0.5 GB) to Drive in cell 2.5.
CKPT_DIR = "/content/ckpts_xlsr_jp_us"

# This Colab image ships a transformers build whose TrainingArguments signature is
# missing some newer kwargs and renames others (eval_strategy vs evaluation_strategy).
# Add each version-sensitive kwarg only if the installed signature accepts it.
_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)

def _add_supported(kwargs, name, value, aliases=()):
    for candidate in (name, *aliases):
        if candidate in _ta_params:
            kwargs[candidate] = value
            return
    print(f"[warn] TrainingArguments has no '{name}' (or aliases {aliases}); skipping.")

# Memory budget for a 15 GB T4 with wav2vec2-large + fp16:
#   - small per-device batch (4) keeps peak self-attention activations small;
#   - gradient_accumulation_steps=4 restores an effective batch of 16;
#   - gradient_checkpointing trades ~25% compute for a big activation-memory saving.
# Combined with the 12 s clip cap (cell 2.3), this fits comfortably.
ta_kwargs = dict(
    output_dir=CKPT_DIR,                # LOCAL disk, not Drive
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch 16
    eval_steps=400,
    save_steps=400,
    logging_steps=50,
    num_train_epochs=15,
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.005,
    fp16=use_cuda,                      # mixed precision on GPU (halves activation memory)
    gradient_checkpointing=True,        # recompute activations in backward -> less VRAM
    save_total_limit=2,
    load_best_model_at_end=True,        # loads best-PER checkpoint into `model` after train
    metric_for_best_model="per",
    greater_is_better=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,        # plain torch datasets aren't column-introspectable
    report_to="none",
)

# Version-sensitive / renamed args:
_add_supported(ta_kwargs, "eval_strategy", "steps", aliases=("evaluation_strategy",))
_add_supported(ta_kwargs, "save_strategy", "steps")
# group_by_length needs an HF-dataset length column, which a plain torch Dataset doesn't
# expose. The 12 s clip cap already bounds worst-case padding, so it's fine to omit.

training_args = TrainingArguments(**ta_kwargs)

# gradient_checkpointing + Trainer: disable the KV/use_cache path to avoid a warning/no-op.
model.config.use_cache = False

# Trainer renamed tokenizer= -> processing_class= in 4.46; pick whichever exists.
_trainer_params = set(inspect.signature(Trainer.__init__).parameters)
_proc_kw = "processing_class" if "processing_class" in _trainer_params else "tokenizer"

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=ComputeMetrics,
    **{_proc_kw: processor},
)
print("Trainer ready. checkpoints -> LOCAL:", CKPT_DIR)
print("effective batch =",
      ta_kwargs["per_device_train_batch_size"] * ta_kwargs["gradient_accumulation_steps"])


## 2.5 Train, save, and evaluate per-language

In [ ]:
import gc
import shutil

# Free VRAM held by a previous attempt. NOTE: empty_cache() only releases PyTorch's unused
# reserved cache — if an earlier failed train() still has live tensors, this won't recover
# them. If "GPU free before train" prints a small number (< ~13 GiB), do Runtime > Restart
# session and re-run from cell 2.0; a wedged CUDA context only truly clears on restart.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU free before train: {(torch.cuda.mem_get_info()[0] / 1024**3):.2f} GiB "
          f"(want > ~13 GiB on a fresh T4)")

train_result = trainer.train()
print(train_result.metrics)

# Checkpoints live on LOCAL disk (CKPT_DIR). load_best_model_at_end=True has already loaded
# the best-PER weights into `model`, so we save ONLY the final model (~0.5 GB, no optimizer
# state) to Drive. This is the sole Drive write, so it won't blow the quota mid-training.
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)          # best-PER weights -> Drive
processor.save_pretrained(OUTPUT_DIR)   # tokenizer/vocab/feature-extractor -> Drive

# Report what landed on Drive and free the (ephemeral) local checkpoints.
def _dir_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024**3

print(f"Saved final model to Drive: {OUTPUT_DIR}  ({_dir_gb(OUTPUT_DIR):.2f} GiB)")
shutil.rmtree(CKPT_DIR, ignore_errors=True)   # local only; frees /content, not Drive
print("Cleared local checkpoints:", CKPT_DIR)


In [ ]:
# Per-language PER on the validation splits.
overall = trainer.evaluate(eval_dataset)
print("Overall eval PER:", round(overall["eval_per"], 4))

for lang in ["us", "jp"]:
    subset_df = eval_df[eval_df["lang"] == lang].reset_index(drop=True)
    subset_ds = SpeechDataset(subset_df)
    metrics = trainer.evaluate(subset_ds, metric_key_prefix=f"eval_{lang}")
    print(f"{lang.upper()} PER:", round(metrics[f"eval_{lang}_per"], 4))


In [ ]:
# Quick qualitative check: transcribe a few eval clips of each language.
model.eval()
device = "cuda" if use_cuda else "cpu"
model.to(device)

def Transcribe(audio_cell):
    array, sr = sf.read(io.BytesIO(audio_cell["bytes"]), dtype="float32")
    inputs = processor(array, sampling_rate=sr, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)[0]
    return IdsToIpa(pred_ids, group_tokens=True)   # CTC decode

for lang in ["us", "jp"]:
    print(f"\n=== {lang.upper()} ===")
    subset_df = eval_df[eval_df["lang"] == lang].reset_index(drop=True)
    for i in range(3):
        row = subset_df.iloc[i]
        print("REF :", row["IPA"])
        print("PRED:", Transcribe(row["audio"]))
        print()
